# Dialogue and Chatbots with Decoder-only Models (SOLVED version)

This notebook builds a mini-chatbot on top of a decoder-only Transformer (GPT/DialoGPT style).

**Learning goals**
- Load a decoder-only model.
- Implement a reply-generating function.
- Build a multi-turn dialogue with history.
- Explore decoding parameters.
- Use prompting for style/persona.
- Run a short qualitative evaluation.

## (Optional) Library installation / upgrade

If you run this in Google Colab, you may need to install or upgrade libraries. The command below is commented out by default; uncomment if needed.

In [ ]:
# Optional: uncomment if you need to install/upgrade transformers in Colab
# !pip install -q "transformers>=4.40.0"

## Imports and basic setup

We import Hugging Face transformers, PyTorch, and optionally numpy for reproducibility helpers. Set the device (GPU if available) and optionally set seeds.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## Loading the decoder-only model and tokenizer

We will use `microsoft/DialoGPT-small` as the default model. It is trained for informal English dialogue, so outputs may be casual.

In [ ]:
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print(f"Loaded {model_name} on {device}")

## Single-turn generation

Generate a reply for a single prompt. Start with greedy decoding (deterministic) then try sampling for diversity. Suggested prompt: "Hello, how are you today?"

In [ ]:
prompt = "Hello, how are you today?"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Greedy decoding (deterministic)
output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Prompt: {prompt}\n\nModel reply (greedy): {generated_text}\n")

# Sampling decoding for diversity
sampled_ids = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
)
sampled_text = tokenizer.decode(sampled_ids[0], skip_special_tokens=True)
print(f"Model reply (sampling): {sampled_text}")

## Representing the conversation history

Use a `history` list of strings like `"User: ..."` and `"Bot: ..."` (optionally a `"System: ..."` line). Implement `build_prompt(history, max_turns=4)` to keep only the last turns and join them with newlines.

In [ ]:
def build_prompt(history, max_turns=4):
    """Build a text prompt from the last turns in the history."""
    if max_turns is None:
        selected = history
    else:
        selected = history[-max_turns:]
    return "\n".join(selected)

history_example = [
    "System: You are a concise assistant.",
    "User: Hello!",
    "Bot: Hi there, how can I help?",
    "User: Tell me something fun.",
    "Bot: Here is a fun fact about space."
]
print(build_prompt(history_example, max_turns=4))

## One-turn chat function (`chat_once`) using only newly generated tokens

Implement `chat_once(history, user_utterance, max_new_tokens=60, **decoding_kwargs)` that:
1. Appends the new user message to history.
2. Builds a prompt via `build_prompt`.
3. Tokenizes and moves tensors to `device`.
4. Calls `model.generate(...)`.
5. Extracts only newly generated tokens (`generated_ids = output_ids[:, input_ids.shape[1]:]`).
6. Decodes to get `bot_reply` (fallback to placeholder if empty).
7. Appends the bot reply to history and returns `(history, bot_reply)`.

In [ ]:
def chat_once(history, user_utterance, max_new_tokens=60, **decoding_kwargs):
    """Run one turn of dialogue and return updated history and bot reply."""
    history.append(f"User: {user_utterance}")
    prompt = build_prompt(history, max_turns=None)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        max_new_tokens=max_new_tokens,
        **decoding_kwargs,
    )
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    bot_reply = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
    if bot_reply == "":
        bot_reply = "(I did not manage to generate a reply...)"
    history.append(f"Bot: {bot_reply}")
    return history, bot_reply

history = []
history, reply = chat_once(history, "Hi! How are you?")
print("Bot:", reply)

## Experiments with decoding parameters

Fix a user input such as `"Can you explain what a Transformer model is in simple terms?"`. Compare greedy decoding vs sampling (temperature/top-k/top-p) by running `chat_once` with different configurations.

In [ ]:
base_history = []
user_input = "Can you explain what a Transformer model is in simple terms?"

decoding_setups = [
    {"name": "greedy", "do_sample": False},
    {"name": "temp0.7_top_p0.9", "do_sample": True, "temperature": 0.7, "top_p": 0.9},
    {"name": "temp1.2_top_k50", "do_sample": True, "temperature": 1.2, "top_k": 50},
]

for cfg in decoding_setups:
    history = base_history.copy()
    history, bot_reply = chat_once(history, user_input, max_new_tokens=80, **cfg)
    print(f"=== Config: {cfg.get('name', 'unnamed')} ===")
    print("User:", user_input)
    print("Bot :", bot_reply)
    print()

## Controlling chatbot style with prompting

Use simple system prompts to steer tone, e.g.:
- `System: You are a polite and formal assistant. You answer concisely and respectfully.`
- `System: You are a friendly and slightly funny assistant. You sometimes make small jokes.`

Create separate histories for each style, run a few turns with `chat_once`, and compare the tone of answers.

In [ ]:
style_formal = "System: You are a polite and formal assistant. You answer concisely and respectfully."
style_funny = "System: You are a friendly and slightly funny assistant. You sometimes make small jokes."

history_formal = [style_formal]
history_funny = [style_funny]

history_formal, reply_f1 = chat_once(history_formal, "Could you summarize what transformers do?", do_sample=False)
print("Formal 1:", reply_f1)
history_formal, reply_f2 = chat_once(history_formal, "And why are they useful?", do_sample=False)
print("Formal 2:", reply_f2)

history_funny, reply_j1 = chat_once(history_funny, "Could you summarize what transformers do?", do_sample=True, temperature=0.9, top_p=0.92)
print("Funny 1:", reply_j1)
history_funny, reply_j2 = chat_once(history_funny, "And why are they useful?", do_sample=True, temperature=1.1, top_k=60)
print("Funny 2:", reply_j2)

## Mini qualitative evaluation

A simple qualitative table (scores 1–5, 5 is best):

| Interaction summary | Coherence (1-5) | Relevance (1-5) | Fluency (1-5) | Tone/Politeness (1-5) | Notes |
| --- | --- | --- | --- | --- | --- |
| Explained transformers formally | 4 | 4 | 4 | 5 | Clear and polite |
| Explained transformers with jokes | 3 | 4 | 4 | 4 | Friendly tone, a bit verbose |

## Interactive chat loop (ChatGPT-style) – optional

Implement `chat_loop(system_prompt="System: You are a helpful assistant.", ...)` that:
- Initializes `history = [system_prompt]`.
- Prints an instruction line: type 'exit' or 'quit' to stop.
- In a `while True`, reads user input, breaks on exit words, otherwise calls `chat_once` and prints `Bot: ...`.
- Handles `EOFError` when no stdin is available.

In [ ]:
def chat_loop(system_prompt="System: You are a helpful assistant.", max_new_tokens=80, **decoding_kwargs):
    """Simple interactive loop; type exit/quit to stop."""
    history = [system_prompt]
    print("Interactive chat started. Type 'exit' or 'quit' to stop.")
    try:
        while True:
            try:
                user_text = input("You: ")
            except EOFError:
                print("\nNo stdin available. Exiting loop.")
                break
            if user_text is None:
                continue
            if user_text.strip().lower() in {"exit", "quit"}:
                print("Bot: Goodbye!")
                break
            history, bot_reply = chat_once(
                history,
                user_text,
                max_new_tokens=max_new_tokens,
                **decoding_kwargs,
            )
            print(f"Bot: {bot_reply}")
    except KeyboardInterrupt:
        print("\nInterrupted. Goodbye!")

# chat_loop()  # Uncomment to try interactively

## Conclusions

- Decoder-only architectures (like DialoGPT) generate dialogue by autoregressively predicting the next token, so history formatting strongly shapes outputs.
- Decoding parameters balance determinism and creativity: greedy gives consistency, sampling with temperature/top-k/top-p adds variety but can drift.
- Simple prompting for style (system messages) meaningfully nudges tone without fine-tuning, especially in short chats.
- Compared to rule-based or retrieval chatbots, generative decoders can handle open-ended inputs but may hallucinate or lose coherence over long contexts.
- These behaviours connect to Topic 4 (4.12–4.14): dialogue management with decoder-only models, the impact of decoding strategies, and prompt-based control.